# Rețele de calculatoare · Săptămâna 2 — Notebook de aplicații (Google Colab)
### Arhitectură și performanță: straturi, modele de referință și cifrele din spatele întârzierii

Notebook însoțitor al **pachetului didactic din Săptămâna 2** (Cursul 2, Seminarul 2, Laboratorul 2 Linux, Laboratorul 2 Windows).

| Partea | În pachetul didactic | În acest notebook |
|---|---|---|
| **A** | Cursul 2 | Construim octet cu octet un cadru real Ethernet/IPv4/TCP/HTTP și îl demultiplexăm; OSI vs. TCP/IP; simulări de cozi (M/M/1, tail drop, bufferbloat); debitul și produsul bandă–întârziere; argumentul capăt-la-capăt ca experiment; mecanismul traceroute |
| **B** | Seminarul 2 | Problemele 1–8 în cod — inclusiv un decodor complet pentru cadrul hexazecimal din Problema 2 și simulări care verifică legea lui Little |
| **C** | Laboratorul 2 (Linux) | Captură de pachete pe VM-ul Colab cu **tcpdump**, analiză cu **tshark** (geamănul în linie de comandă al lui Wireshark) și **Scapy**; fișierele de captură pot fi descărcate și deschise în Wireshark |
| **D** | Laboratorul 2 (Windows) | `tracert` vs. `traceroute` văzute în pachete, Linux/Windows în paralel, fișiere pentru Wireshark pe Windows |

**Cum se folosește.** Rulați celulele în ordine. Celulele marcate cu ✍️ sunt pentru răspunsurile voastre la Q1–Q16, astfel încât notebook-ul să fie raportul vostru (`File → Download → .ipynb`), împreună cu fișierul de captură `web.pcapng` produs în Sarcina 3.

> **Reguli de conduită.** Capturați doar pe propria mașină (aici: propriul VM Colab) și doar propriul trafic.

## 0 · Pregătire

In [ ]:
# ---------------------------------------------------------------
#  Funcții ajutătoare folosite în tot notebook-ul. Rulați întâi această celulă.
# ---------------------------------------------------------------
import os, re, sys, time, math, json, socket, shutil, subprocess, threading, ipaddress
from collections import deque, Counter

IN_COLAB = 'google.colab' in sys.modules
SUDO = '' if os.geteuid() == 0 else 'sudo '

def sh(cmd, timeout=60, show=True):
    """Rulează o comandă shell, afișează ieșirea (stdout+stderr), nu ridică niciodată excepții."""
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        out = (r.stdout or '') + (r.stderr or '')
    except subprocess.TimeoutExpired as e:
        o = e.stdout or ''
        out = (o.decode(errors='replace') if isinstance(o, bytes) else o) + f'\n[oprit după {timeout} s]'
    if show:
        print(f'$ {cmd}\n{out.rstrip()}\n')
    return out

def have(tool):
    return shutil.which(tool) is not None

def default_route():
    """(interfața, gateway-ul) folosite pentru a ajunge în Internet."""
    out = sh('ip route get 1.1.1.1', show=False)
    dev = re.search(r'\bdev (\S+)', out)
    via = re.search(r'\bvia (\S+)', out)
    src = re.search(r'\bsrc (\S+)', out)
    return (dev.group(1) if dev else 'any', via.group(1) if via else None, src.group(1) if src else None)

def resolve(host):
    try:
        return socket.gethostbyname(host)
    except Exception as e:
        print(f'nu pot rezolva {host}: {e}')
        return None

def tcp_rtt(host, port=443, n=5, timeout=3):
    """Măsoară RTT ca durată a unui connect() TCP (un schimb SYN / SYN-ACK).
    Funcționează și acolo unde ping-ul ICMP este blocat (ca pe multe VM-uri din cloud)."""
    ip = resolve(host)
    if not ip:
        return []
    rtts = []
    for _ in range(n):
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(timeout)
        t0 = time.perf_counter()
        try:
            s.connect((ip, port))
            rtts.append((time.perf_counter() - t0) * 1000)
        except Exception as e:
            print(f'  conectarea la {ip}:{port} a eșuat: {e}')
        finally:
            s.close()
        time.sleep(0.2)
    if rtts:
        print(f'RTT prin conectare TCP la {host} ({ip}:{port}): min/med/max = '
              f'{min(rtts):.2f}/{sum(rtts)/len(rtts):.2f}/{max(rtts):.2f} ms  ({len(rtts)}/{n} răspunsuri)')
    return rtts

def fmt_time(seconds):
    """Afișează frumos o durată."""
    s = abs(seconds)
    if s < 1e-6: return f'{seconds*1e9:.3g} ns'
    if s < 1e-3: return f'{seconds*1e6:.4g} µs'
    if s < 1:    return f'{seconds*1e3:.4g} ms'
    if s < 120:  return f'{seconds:.4g} s'
    if s < 7200: return f'{seconds/60:.4g} min'
    if s < 2*86400: return f'{seconds/3600:.4g} h'
    return f'{seconds/86400:.4g} zile'

def check(name, got, expected, rel=0.02):
    ok = math.isclose(got, expected, rel_tol=rel, abs_tol=1e-12)
    print(f"{'✅' if ok else '❌'} {name}: calculat {got:.6g}, așteptat ≈ {expected:.6g}")
    return ok

print('Funcțiile ajutătoare au fost încărcate. Rulează în Colab:', IN_COLAB, '| root:', os.geteuid() == 0)

In [ ]:
# Unelte de linie de comandă (tcpdump, tshark, traceroute, dig, curl) și Scapy. Circa 1 minut în Colab.
sh(f"echo 'wireshark-common wireshark-common/install-setuid boolean false' | {SUDO}debconf-set-selections", show=False)
sh(f'{SUDO}apt-get -qq update > /dev/null 2>&1; DEBIAN_FRONTEND=noninteractive {SUDO}apt-get -qq install -y '
   'tcpdump tshark traceroute dnsutils curl iproute2 iputils-ping > /dev/null 2>&1; echo apt done', timeout=900)
sh(f'{sys.executable} -m pip -q install scapy 2>&1 | tail -1', timeout=300)
for t in ['tcpdump', 'tshark', 'editcap', 'traceroute', 'dig', 'curl']:
    print(f'{t:11s}', '✅' if have(t) else '❌ lipsește')

In [ ]:
import struct, zlib, hashlib, random, signal, shlex
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 4.5), 'axes.grid': True, 'grid.alpha': .3})
K, M, G = 1e3, 1e6, 1e9

def hexdump(data, width=16):
    for off in range(0, len(data), width):
        chunk = data[off:off + width]
        hx = ' '.join(f'{b:02x}' for b in chunk)
        asc = ''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk)
        print(f'{off:04x}  {hx:<{width*3}} {asc}')

---
# Partea A · Cursul 2 — cursul în cod

## A.1 Serviciu, interfață, protocol
**Serviciul** TCP este un flux de octeți fiabil; **interfața** sa sunt apelurile socket; **protocolul** său sunt segmentele schimbate cu entitatea pereche. Mai jos, două procese de pe aceeași mașină folosesc doar interfața — nu văd niciodată un segment.

In [ ]:
srv = socket.socket(); srv.bind(('127.0.0.1', 0)); srv.listen(1)
port = srv.getsockname()[1]

def echo_server():
    conn, _ = srv.accept()                     # interfață: accept
    data = conn.recv(1024)                     # interfață: recv
    conn.sendall(data.upper())                 # interfață: send
    conn.close()

threading.Thread(target=echo_server, daemon=True).start()
c = socket.create_connection(('127.0.0.1', port))   # interfață: connect (handshake-ul TCP în 3 pași are loc dedesubt)
c.sendall(b'hello, transport layer')
print('răspuns:', c.recv(1024)); c.close(); srv.close()

## A.2 Încapsularea, octet cu octet
Construim cadrul din **Cursul 2, secțiunea 3.1**: o cerere HTTP de 400 de octeți în TCP (20 B), IPv4 (20 B) și Ethernet (antet de 14 B + trailer de 4 B). Fiecare sumă de control este calculată exact ca în gazdele reale.

In [ ]:
def inet_checksum(data: bytes) -> int:
    """Suma de control Internet (RFC 1071): suma în complement față de unu a cuvintelor de 16 biți."""
    if len(data) % 2:
        data += b'\0'
    s = sum(struct.unpack(f'!{len(data)//2}H', data))
    while s >> 16:
        s = (s & 0xFFFF) + (s >> 16)
    return ~s & 0xFFFF

def mac(s): return bytes.fromhex(s.replace(':', ''))
def ip(s):  return socket.inet_aton(s)

def tcp_segment(src_ip, dst_ip, sport, dport, payload=b'', seq=1, ack=1, flags=0x18, window=64240):
    hdr = struct.pack('!HHIIBBHHH', sport, dport, seq, ack, 5 << 4, flags, window, 0, 0)
    pseudo = struct.pack('!4s4sBBH', ip(src_ip), ip(dst_ip), 0, 6, len(hdr) + len(payload))
    csum = inet_checksum(pseudo + hdr + payload)
    return hdr[:16] + struct.pack('!H', csum) + hdr[18:] + payload

def udp_datagram(src_ip, dst_ip, sport, dport, payload):
    length = 8 + len(payload)
    hdr = struct.pack('!HHHH', sport, dport, length, 0)
    pseudo = struct.pack('!4s4sBBH', ip(src_ip), ip(dst_ip), 0, 17, length)
    return struct.pack('!HHHH', sport, dport, length, inet_checksum(pseudo + hdr + payload)) + payload

def ipv4_packet(src_ip, dst_ip, proto, payload, ttl=64, ident=0x1c46):
    hdr = struct.pack('!BBHHHBBH4s4s', 0x45, 0, 20 + len(payload), ident, 0x4000, ttl, proto, 0, ip(src_ip), ip(dst_ip))
    return hdr[:10] + struct.pack('!H', inet_checksum(hdr)) + hdr[12:] + payload

def ethernet_frame(dst_mac, src_mac, ethertype, payload, with_fcs=True):
    frame = mac(dst_mac) + mac(src_mac) + struct.pack('!H', ethertype) + payload
    return frame + struct.pack('<I', zlib.crc32(frame)) if with_fcs else frame

# O cerere HTTP de 400 de octeți
lines = ['GET /index.html HTTP/1.1', 'Host: www.example.org', 'User-Agent: course-notebook/1.0', 'Accept: */*']
base = '\r\n'.join(lines) + '\r\n'
pad = 400 - len(base) - len('X-Padding: \r\n\r\n')
http = (base + 'X-Padding: ' + 'x' * pad + '\r\n\r\n').encode()

CLIENT_MAC, GW_MAC = '52:54:00:3a:7f:10', '52:54:00:12:34:01'
CLIENT_IP, SERVER_IP = '10.20.3.57', '93.184.215.14'
seg = tcp_segment(CLIENT_IP, SERVER_IP, 51234, 80, http)
pkt = ipv4_packet(CLIENT_IP, SERVER_IP, 6, seg)
frame = ethernet_frame(GW_MAC, CLIENT_MAC, 0x0800, pkt)

for name, size in [('cerere HTTP', len(http)), ('+ antet TCP', len(seg)), ('+ antet IPv4', len(pkt)), ('+ antet și trailer Ethernet', len(frame))]:
    print(f'{name:32s} {size:4d} octeți')
print(f'Antete: {len(frame) - len(http)} din {len(frame)} octeți\n')
hexdump(frame[:80]); print('...')

In [ ]:
# Același cadru decodat de o implementare independentă (Scapy), ca să verificăm octeții și sumele de control
from scapy.all import Ether, IP as sIP, TCP as sTCP
f = Ether(frame[:-4])                       # uneltele de captură omit de obicei FCS-ul de 4 octeți
print(f.summary())
g = f.copy(); del g[sIP].chksum; del g[sTCP].chksum; g = Ether(bytes(g))   # Scapy le recalculează
print(f'Sumă control IP : a noastră 0x{f[sIP].chksum:04x}   Scapy 0x{g[sIP].chksum:04x}')
print(f'Sumă control TCP: a noastră 0x{f[sTCP].chksum:04x}   Scapy 0x{g[sTCP].chksum:04x}')
print('FCS (CRC-32) :', frame[-4:].hex())
print('Verificare MTU: încărcătura TCP maximă peste Ethernet =', 1500 - 20 - 20, 'octeți')

### Demultiplexarea
La fiecare strat, receptorul citește **un singur câmp** ca să decidă cine primește încărcătura: EtherType → IP Protocol → portul destinație.

In [ ]:
ETHERTYPE = {0x0800: 'IPv4', 0x86DD: 'IPv6', 0x0806: 'ARP'}
IPPROTO = {1: 'ICMP', 6: 'TCP', 17: 'UDP'}
PORTS = {53: 'DNS', 80: 'HTTP', 443: 'HTTPS', 22: 'SSH', 25: 'SMTP'}

def port_class(p):
    return 'bine cunoscut' if p < 1024 else 'înregistrat' if p < 49152 else 'dinamic'

def demultiplex(frame):
    et = struct.unpack('!H', frame[12:14])[0]
    print(f'Legătura  citește EtherType 0x{et:04x}  → {ETHERTYPE.get(et, "?")}')
    if et != 0x0800: return
    ihl = (frame[14] & 0x0F) * 4
    proto = frame[14 + 9]
    print(f'Rețeaua   citește Protocol  {proto:6d}  → {IPPROTO.get(proto, "?")}')
    l4 = frame[14 + ihl:]
    sport, dport = struct.unpack('!HH', l4[:4])
    print(f'Transport citește port dst {dport:6d}  → {PORTS.get(dport, "?")}  '
          f'(portul sursă {sport} este {port_class(sport)}, portul destinație {dport} este {port_class(dport)})')
demultiplex(frame)

### Ce adrese se schimbă la fiecare hop?
Adresa **IP** destinație este cea a serverului pe tot drumul; adresa **MAC** destinație este doar următorul hop. Fiecare router construiește un cadru nou și scade TTL-ul.

In [ ]:
path = [  # (nod, MAC-ul interfeței din stânga, MAC-ul din dreapta)
    ('client', None, '52:54:00:3a:7f:10'),
    ('R1 (gateway)', '52:54:00:12:34:01', 'aa:00:00:00:01:02'),
    ('R2', 'aa:00:00:00:02:01', 'aa:00:00:00:02:02'),
    ('server', 'bb:00:00:00:00:80', None),
]
ttl = 64
print(f'{"legătura":24s} {"src MAC":18s} {"dst MAC":18s} {"src IP":12s} {"dst IP":14s} TTL')
for (a, _, a_out), (z, z_in, _) in zip(path, path[1:]):
    print(f'{a + " → " + z:24s} {a_out:18s} {z_in:18s} {CLIENT_IP:12s} {SERVER_IP:14s} {ttl}')
    ttl -= 1

## A.3 Modele de referință

In [ ]:
OSI = [(7, 'Aplicație', 'HTTP, SMTP, DNS'), (6, 'Prezentare', 'UTF-8, JSON, TLS — în aplicații/biblioteci'),
       (5, 'Sesiune', 'sesiuni HTTP, reluare TLS — în aplicații'), (4, 'Transport', 'TCP, UDP, QUIC'),
       (3, 'Rețea', 'IP'), (2, 'Legătură date', 'Ethernet, Wi-Fi'), (1, 'Fizic', 'semnalizare pe cupru, fibră, radio')]
FIVE = {7: 'Aplicație', 6: 'Aplicație', 5: 'Aplicație', 4: 'Transport', 3: 'Rețea', 2: 'Legătură', 1: 'Fizic'}
RFC1122 = {7: 'Aplicație', 6: 'Aplicație', 5: 'Aplicație', 4: 'Transport', 3: 'Internet', 2: 'Legătură', 1: 'Legătură'}
print(f'{"#":2s} {"OSI (7)":13s} {"5 straturi":12s} {"RFC 1122 (4)":13s} unde se află în Internet')
for n, name, where in OSI:
    print(f'{n:<2d} {name:13s} {FIVE[n]:12s} {RFC1122[n]:13s} {where}')
print('\nMnemonic (în engleză), de la stratul 1 în sus: „Please Do Not Throw Sausage Pizza Away”')

In [ ]:
# Talia îngustă (Figura 2)
tops = ['HTTP', 'SMTP', 'DNS', 'SSH', 'RTP', 'aplicații QUIC']; mids = ['TCP', 'UDP', 'QUIC']
bots = ['Ethernet', 'Wi-Fi', '4G/5G', 'DSL', 'fibră', 'satelit']
fig, ax = plt.subplots(figsize=(8, 4.5)); ax.axis('off')
def row(items, y, w, color):
    xs = np.linspace(-w, w, len(items))
    for x, t in zip(xs, items):
        ax.text(x, y, t, ha='center', va='center', bbox=dict(boxstyle='round', fc=color), fontsize=9)
row(tops, 4, 3.3, '#ffe0b3'); row(mids, 3, 1.6, '#cfe3ff'); row(['IP'], 2, 0, '#ffb3b3'); row(bots, 1, 3.3, '#d5f5d5')
ax.plot([-3.8, -0.4, -3.8], [4.4, 2, 0.6], 'k--', lw=.8); ax.plot([3.8, 0.4, 3.8], [4.4, 2, 0.6], 'k--', lw=.8)
ax.set_xlim(-4, 4); ax.set_ylim(0.4, 4.6); ax.set_title('Clepsidra: un singur protocol de nivel rețea la mijloc'); plt.show()

## A.4 Cine face regulile
IETF (Internet-Drafts → RFC-uri: *Proposed Standard* → *Internet Standard*), IEEE 802 (Ethernet, Wi-Fi), ITU-T și 3GPP (telecomunicații, 4G/5G), W3C și WHATWG (Web), IANA și RIR-urile (numere; România: RIPE NCC). *„We reject kings, presidents and voting. We believe in rough consensus and running code.”* (D. Clark, 1992)

## A.5 Întârzierea în coadă în detaliu
Intensitatea traficului `ρ = L·a / R`. În modelul M/M/1, timpul mediu petrecut în router este `d = (L/R) / (1 − ρ)`. Verificăm formula printr-o **simulare** cu sosiri și lungimi de pachet aleatoare (recurența lui Lindley: fiecare pachet așteaptă cât i-a lăsat cel de dinainte).

In [ ]:
def mm1_simulate(rho, n=200_000, service_mean=1.0, seed=0):
    """Întoarce (timpi de așteptare, timpi de ședere, momente de sosire) pentru o coadă M/M/1."""
    rng = np.random.default_rng(seed)
    inter = rng.exponential(service_mean / rho, n)
    serv = rng.exponential(service_mean, n)
    arrivals = np.cumsum(inter)
    wait = np.zeros(n)
    for i in range(1, n):
        wait[i] = max(0.0, wait[i-1] + serv[i-1] - inter[i])
    return wait, wait + serv, arrivals

rhos = [0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95]
sim = [mm1_simulate(r)[1].mean() for r in rhos]
grid = np.linspace(0, 0.97, 300)
plt.plot(grid, 1 / (1 - grid), label='formula M/M/1  (L/R)/(1−ρ)')
plt.plot(rhos, sim, 'o', label='simulare (200 000 de pachete)')
plt.ylim(0, 25); plt.xlabel('intensitatea traficului ρ'); plt.ylabel('întârzierea medie (în unități L/R)')
plt.title('Întârzierea explodează când ρ → 1'); plt.legend(); plt.show()
for r, s in zip(rhos, sim):
    print(f'ρ = {r:4.2f}: formula {1/(1-r):6.2f} · L/R   simularea {s:6.2f} · L/R')

### Pierderi cu buffere finite și bufferbloat
Un buffer de **K** pachete aruncă sosirile când este plin (*tail drop*). Al doilea grafic arată de ce bufferele uriașe nu sunt soluția: în suprasarcină, un buffer plin este pur și simplu o coadă lungă, în spatele căreia trebuie să aștepte fiecare pachet.

In [ ]:
def finite_queue(rho, K, n=100_000, seed=1, service_mean=1.0):
    """M/M/1/K prin simulare de evenimente. Întoarce (fracția pierdută, întârzierea medie a pachetelor livrate)."""
    rng = np.random.default_rng(seed)
    t, departures, lost, delays = 0.0, deque(), 0, []
    for _ in range(n):
        t += rng.exponential(service_mean / rho)
        while departures and departures[0] <= t:
            departures.popleft()
        if len(departures) >= K + 1:          # K în așteptare + 1 în transmisie
            lost += 1; continue
        start = departures[-1] if departures else t
        dep = max(start, t) + rng.exponential(service_mean)
        departures.append(dep); delays.append(dep - t)
    return lost / n, float(np.mean(delays))

rho_axis = np.linspace(0.5, 1.3, 9)
for K in (5, 20, 100):
    plt.plot(rho_axis, [finite_queue(r, K, n=40_000)[0] for r in rho_axis], 'o-', label=f'buffer K = {K}')
plt.xlabel('intensitatea traficului ρ'); plt.ylabel('fracția de pachete pierdute'); plt.legend()
plt.title('Pierderi tail-drop: peste ρ = 1 nicio dimensiune de buffer nu ajută'); plt.show()

In [ ]:
Ks = [5, 10, 20, 50, 100, 200, 500, 1000]
tx_ms = 1500 * 8 / 10e6 * 1e3            # un pachet de 1500 de octeți pe o legătură de upload casnică de 10 Mbit/s = 1,2 ms
res = [finite_queue(1.05, K, n=60_000) for K in Ks]
plt.semilogx(Ks, [d * tx_ms for _, d in res], 'o-')
plt.xlabel('dimensiunea bufferului (pachete)'); plt.ylabel('întârzierea medie (ms)')
plt.title('Bufferbloat: un upload la ρ = 1,05 pe o legătură de 10 Mbit/s'); plt.show()
for K, (loss, d) in zip(Ks, res):
    print(f'K = {K:5d}: pierderi {loss:5.1%}  întârziere {d*tx_ms:7.1f} ms  ← adăugată fiecărui pachet, inclusiv apelului video')

## A.6 Debitul și produsul bandă–întârziere
`debit = min(R₁, …, Rₙ)` (sau partea echitabilă dintr-o legătură partajată) și `BDP = R × RTT` — datele care trebuie să fie „în zbor” ca să țină calea plină. Un emițător căruia i se permit doar **W** octeți neconfirmați atinge cel mult `W / RTT`.

In [ ]:
def bdp_bytes(R_bps, rtt_s): return R_bps * rtt_s / 8
def window_limited(W_bytes, rtt_s, R_bps): return min(R_bps, W_bytes * 8 / rtt_s)

b = bdp_bytes(1*G, 0.030)
print(f'BDP pentru 1 Gbit/s × 30 ms = {b*8/1e6:.0f} Mbit = {b/1e6:.2f} MB')
print(f'Cu o fereastră de 64 KB: {window_limited(65535, 0.030, 1*G)/1e6:.1f} Mbit/s pe o cale de 1 Gbit/s')

rtts = np.linspace(0.001, 0.3, 300)
for W in (65535, 1<<20, 16<<20):
    plt.plot(rtts * 1e3, [window_limited(W, r, 1*G) / 1e6 for r in rtts], label=f'fereastră {W/1024:.0f} KB')
plt.xlabel('RTT (ms)'); plt.ylabel('debit maxim (Mbit/s)'); plt.yscale('log'); plt.legend()
plt.title('O cale de 1 Gbit/s: fereastra, nu legătura, limitează transferurile la distanță'); plt.show()

### Întrebarea de la început
Voi: legături de 1 Gbit/s, 10 Gbit/s, 100 Mbit/s, RTT 10 ms. Vecinul: 1 Gbit/s peste tot, RTT 300 ms. Un model simplu al unei descărcări TCP: un RTT pentru handshake, apoi o fereastră care pornește de la 10 segmente și se dublează la fiecare RTT (*slow start*) până umple calea.

In [ ]:
def download_time(size_bytes, R_bps, rtt, mss=1460, init_segments=10):
    t, sent, cwnd = rtt, 0.0, init_segments * mss        # întâi handshake-ul
    full = bdp_bytes(R_bps, rtt)
    while True:
        per_rtt = min(cwnd, full)
        if size_bytes - sent <= per_rtt:
            return t + rtt / 2 + (size_bytes - sent) * 8 / R_bps
        sent += per_rtt; t += rtt; cwnd *= 2

sizes = np.logspace(4, 10.5, 200)
you = [download_time(s, 100*M, 0.010) for s in sizes]
nb_ = [download_time(s, 1*G, 0.300) for s in sizes]
plt.loglog(sizes / 1e6, you, label='voi: gâtuire 100 Mbit/s, RTT 10 ms')
plt.loglog(sizes / 1e6, nb_, label='vecinul: 1 Gbit/s, RTT 300 ms')
plt.xlabel('dimensiunea fișierului (MB)'); plt.ylabel('timpul de descărcare (s)'); plt.legend()
plt.title('Fișiere mici: decide întârzierea · fișiere mari: decide debitul'); plt.show()
for s in (1e6, 10e9):
    print(f'{s/1e6:>8.0f} MB: voi {download_time(s, 100*M, .01):8.2f} s   vecinul {download_time(s, 1*G, .3):8.2f} s')

## A.7 Argumentul capăt-la-capăt, ca experiment
Un fișier traversează trei hop-uri. Fiecare legătură verifică un CRC și retransmite cadrele corupte, deci fiecare hop este „fiabil”. Dar un router corupe ocazional un octet **în memoria sa**, după verificarea pe legătură. Doar o sumă de control calculată de emițător peste întregul fișier și verificată de receptor prinde asta.

In [ ]:
def transfer(file_bytes, hops=3, link_ber=1e-5, router_bug=0.002, frame=1000, rng=None):
    rng = rng or random.Random(0)
    frames = [bytearray(file_bytes[i:i+frame]) for i in range(0, len(file_bytes), frame)]
    link_retx = 0
    for hop in range(hops):
        for k, f in enumerate(frames):
            while True:                                       # fiabilitate hop cu hop: CRC + retransmitere
                sent_crc = zlib.crc32(f)
                rx = bytearray(f)
                if rng.random() < link_ber * len(f) * 8:      # eroare de bit pe fir
                    rx[rng.randrange(len(rx))] ^= 1 << rng.randrange(8)
                if zlib.crc32(rx) == sent_crc: break
                link_retx += 1
            if hop < hops - 1 and rng.random() < router_bug:  # corupție în interiorul routerului, după verificare
                rx[rng.randrange(len(rx))] ^= 0xFF
            frames[k] = rx
    return bytes(b''.join(frames)), link_retx

rng = random.Random(42)
data = os.urandom(1_000_000)
sender_digest = hashlib.sha256(data).hexdigest()
bad = 0
for i in range(10):
    received, retx = transfer(data, rng=rng)
    ok = hashlib.sha256(received).hexdigest() == sender_digest
    bad += not ok
    print(f'transferul {i+1:2d}: {retx:3d} retransmiteri pe legături, toate verificările pe legături au trecut → verificarea capăt-la-capăt: {"OK" if ok else "CORUPT"}')
print(f'\n{bad}/10 fișiere au fost corupte, deși fiecare hop era „fiabil”.')
print('Soluția: receptorul compară SHA-256 cu cel al emițătorului și cere din nou fișierul (reîncercare capăt-la-capăt).')

## A.8 Cum funcționează traceroute
Probe cu TTL = 1, 2, 3, … Fiecare router scade TTL-ul și, când ajunge la 0, aruncă proba și întoarce **ICMP Time Exceeded (tipul 11)**. Destinația răspunde cu **Port Unreachable** (tipul 3, codul 3 — probele UDP din Linux) sau **Echo Reply** (tipul 0 — `tracert` din Windows).

In [ ]:
def traceroute_sim(routers, dest, probes_per_hop=3, style='linux'):
    sent = 0
    for ttl in range(1, len(routers) + 2):
        sent += probes_per_hop
        t = ttl
        for node in routers:                 # fiecare router scade TTL-ul
            t -= 1
            if t == 0:
                print(f'TTL={ttl:2d}: expiră la   {node:12s} → ICMP Time Exceeded (tipul 11)'); break
        else:
            reply = 'ICMP Port Unreachable (tipul 3)' if style == 'linux' else 'ICMP Echo Reply (tipul 0)'
            print(f'TTL={ttl:2d}: ajunge la  {dest:12s} → {reply}'); break
    return sent

n = traceroute_sim(['gw', 'core1', 'nren-1', 'nren-2', 'transit-1'], 'server')
print('probe trimise:', n)
print('\nExercițiul 5: 14 hop-uri × 3 probe =', 14 * 3, 'probe; hop-urile 1–13 trimit Time Exceeded, hop-ul 14 Port Unreachable / Echo Reply')

## A.9 Exercițiile de la sfârșitul cursului, verificate prin cod

In [ ]:
print('1. mesaj de 20 de octeți peste UDP (8) + IPv6 (40) + Ethernet (18)')
check('   fracția de date ale aplicației', 20 / (20 + 8 + 40 + 18), 0.23, rel=0.02)
print('2. pachete de 1000 de octeți, legătură de 8 Mbit/s')
LR = 8000 / 8e6
check('   rata de sosire pentru ρ = 1 (pachete/s)', 1 / LR, 1000)
for a, exp in [(500, 2e-3), (800, 5e-3), (950, 20e-3)]:
    rho = a * LR
    check(f'   întârzierea la {a} pachete/s (ρ = {rho:.2f}) (s)', LR / (1 - rho), exp)
print('3. 100 / 40 / 1000 Mbit/s, ultima partajată de 50 de transferuri')
check('   debit (Mbit/s)', min(100, 40, 1000 / 50), 20)
print('4. BDP pentru 10 Gbit/s × 80 ms')
check('   MB', bdp_bytes(10*G, 0.080) / 1e6, 100)
print('5. traceroute, 14 hop-uri, 3 probe pe hop')
check('   probe', 14 * 3, 42)

**Întrebări de recapitulare**: (1) serviciu/interfață/protocol la un alt strat · (2) la ce protocoale participă routerul din Figura 1 · (3) cele trei câmpuri citite pentru a livra un cadru unui browser · (4) de ce se schimbă MAC-ul la fiecare hop, iar IP-ul nu · (5) straturi OSI fără corespondent în TCP/IP · (6) talia îngustă: ușor de adăugat aplicații, greu de schimbat IP-ul · (7) „60 % folosit, 40 % liber, nu trebuie modernizat?” · (8) de ce un buffer mai mare nu vindecă pierderile · (9) argumentul capăt-la-capăt într-o propoziție.

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **R1:** 
- **R2:** 
- **R3:** 
- **R4:** 
- **R5:** 
- **R6:** 
- **R7:** 
- **R8:** 
- **R9:** 

---
# Partea B · Seminarul 2 — Antete, cozi și debit

## B.1 Problema 1 — ce strat?
Scrieți răspunsurile voastre în `MINE`, apoi rulați celula.

In [ ]:
FUNCTIONS = {
    'a': 'Alegerea legăturii de ieșire pentru un pachet adresat altei țări',
    'b': 'Conversia unui document dintr-o codificare de caractere în alta',
    'c': 'Retransmiterea unui segment a cărui confirmare nu a sosit',
    'd': 'Reprezentarea unui 1 ca tranziție de tensiune pe un cablu',
    'e': 'Detectarea faptului că un cadru a fost corupt pe o legătură Wi-Fi',
    'f': 'Reluarea unei descărcări lungi de unde s-a oprit după o deconectare',
    'g': 'Livrarea unui segment sosit procesului potrivit de pe o gazdă',
    'h': 'Decizia care dintre gazdele de pe un cablu partajat poate transmite acum',
}
SOLUTION = {'a': ('rețea', 3), 'b': ('aplicație', 6), 'c': ('transport', 4), 'd': ('fizic', 1),
            'e': ('legătură', 2), 'f': ('aplicație', 5), 'g': ('transport', 4), 'h': ('legătură', 2)}
MINE = {'a': ('rețea', 3), 'b': ('', 0), 'c': ('', 0), 'd': ('', 0), 'e': ('', 0), 'f': ('', 0), 'g': ('', 0), 'h': ('', 0)}

score = 0
for k, text in FUNCTIONS.items():
    ok = MINE[k] == SOLUTION[k]; score += ok
    print(f"{'✅' if ok else '❌'} ({k}) {text[:62]:62s} → 5 straturi: {SOLUTION[k][0]:11s} OSI: {SOLUTION[k][1]}")
print(f'\nScor: {score}/8')

## B.2 Problema 2 — decodarea unui cadru de mână (și prin cod)
Decodorul de mai jos citește fiecare câmp exact după structura descrisă în seminar. Încercați întâi pe hârtie.

In [ ]:
HEX = """
0000  52 54 00 12 34 01 52 54 00 3a 7f 10 08 00 45 00
0010  00 3d 1c 46 40 00 40 11 06 d5 0a 14 03 39 0a 14
0020  00 35 c8 22 00 35 00 29 96 d2 3b 7a 01 00 00 01
0030  00 00 00 00 00 00 03 77 77 77 07 65 78 61 6d 70
0040  6c 65 03 6f 72 67 00 00 01 00 01
"""
def from_dump(text):
    return bytes.fromhex(''.join(' '.join(l.split()[1:]) for l in text.strip().splitlines()))

def fmt_mac(b): return ':'.join(f'{x:02x}' for x in b)

def decode_ethernet(f):
    d = {'dst_mac': fmt_mac(f[0:6]), 'src_mac': fmt_mac(f[6:12]), 'ethertype': struct.unpack('!H', f[12:14])[0]}
    return d, f[14:]

def decode_ipv4(p):
    v_ihl, tos, total, ident, flags_frag, ttl, proto, csum = struct.unpack('!BBHHHBBH', p[:12])
    ihl = (v_ihl & 0x0F) * 4
    d = {'version': v_ihl >> 4, 'header_len': ihl, 'total_len': total, 'id': ident,
         'DF': bool(flags_frag & 0x4000), 'ttl': ttl, 'protocol': proto, 'checksum': csum,
         'src': socket.inet_ntoa(p[12:16]), 'dst': socket.inet_ntoa(p[16:20]),
         'checksum_ok': inet_checksum(p[:ihl]) == 0}       # suma unui antet corect dă 0
    return d, p[ihl:total]

def decode_udp(u):
    sport, dport, length, csum = struct.unpack('!HHHH', u[:8])
    return {'sport': sport, 'dport': dport, 'length': length, 'checksum': csum}, u[8:length]

def decode_tcp(t):
    sport, dport, seq, ack, off, flags, win, csum, urg = struct.unpack('!HHIIBBHHH', t[:20])
    hl = (off >> 4) * 4
    names = [n for bit, n in [(0x02, 'SYN'), (0x10, 'ACK'), (0x01, 'FIN'), (0x04, 'RST'), (0x08, 'PSH')] if flags & bit]
    return {'sport': sport, 'dport': dport, 'seq': seq, 'ack': ack, 'header_len': hl,
            'flags': '+'.join(names), 'window': win, 'options_len': hl - 20}, t[hl:]

def dns_qname(msg, off=12):
    labels = []
    while msg[off]:
        n = msg[off]; labels.append(msg[off+1:off+1+n].decode()); off += 1 + n
    qtype, qclass = struct.unpack('!HH', msg[off+1:off+5])
    return '.'.join(labels), qtype, qclass

frame2 = from_dump(HEX)
eth, rest = decode_ethernet(frame2)
ip4, rest = decode_ipv4(rest)
udp, dns = decode_udp(rest)
print('1.', f"dst MAC {eth['dst_mac']}  src MAC {eth['src_mac']}  EtherType 0x{eth['ethertype']:04x} → {ETHERTYPE[eth['ethertype']]}")
print('2.', f"versiunea IP {ip4['version']}, antet {ip4['header_len']} B, lungime totală {ip4['total_len']} B; "
      f"cadrul afișat = {len(frame2)} B = 14 + {len(frame2) - 14}  → consecvent: {len(frame2) - 14 == ip4['total_len']}")
print('3.', f"TTL {ip4['ttl']}, protocol {ip4['protocol']} = {IPPROTO[ip4['protocol']]}, {ip4['src']} → {ip4['dst']}  "
      f"(suma de control a antetului 0x{ip4['checksum']:04x} validă: {ip4['checksum_ok']})")
print('4.', f"porturi {udp['sport']} → {udp['dport']} ({port_class(udp['sport'])} → {PORTS[udp['dport']]}); "
      f"lungime UDP {udp['length']} = 8 + {len(dns)} octeți de DNS")
name, qtype, qclass = dns_qname(dns)
print('5.', f"numele căutat: {name}   tipul {qtype} (A)  clasa {qclass} (IN)")
same = ipaddress.ip_network(ip4['src'] + '/24', strict=False) == ipaddress.ip_network(ip4['dst'] + '/24', strict=False)
print('6.', f"{ip4['src']}/24 și {ip4['dst']}/24 în aceeași subrețea? {same} → cadrul merge la "
      f"{'serverul DNS direct' if same else 'gateway-ul implicit, deci MAC-ul destinație este al gateway-ului'}")

## B.3 Problema 3 — cât din legătură este util?
Pe fir: preambul 8 B + antet 14 B + încărcătură + FCS 4 B + spațiu între cadre 12 B. Încărcăturile mai mici de 46 B sunt completate până la 46.

In [ ]:
def wire_bytes(payload):
    return 8 + 14 + max(payload, 46) + 4 + 12

check('1. timpi-octet ai unui cadru complet', wire_bytes(1500), 1538)
check('2. debit util TCP/IPv4 pe 100 Mbit/s (Mbit/s)', 1460 / wire_bytes(1500) * 100, 94.9, rel=0.002)
check('3. debit util TCP/IPv6 (Mbit/s)', 1440 / wire_bytes(1500) * 100, 93.6, rel=0.002)
check('4. fracția audio, eșantioane de 20 de octeți peste UDP/IPv4', 20 / wire_bytes(20 + 8 + 20), 20 / 86)

app = np.arange(1, 1473)
plt.plot(app, app / np.array([wire_bytes(a + 28) for a in app]) * 100)
plt.xlabel('octeți de aplicație per pachet UDP/IPv4'); plt.ylabel('partea utilă dintr-o legătură de 100 Mbit/s (%)')
plt.title('Mesajele mici irosesc legătura'); plt.show()

## B.4 Problema 4 — intensitatea traficului (legătură de 10 Mbit/s, pachete de 1250 de octeți, M/M/1)

In [ ]:
LR = 1250 * 8 / 10e6
check('1. L/R (s)', LR, 1e-3); check('1. rata de sosire pentru ρ = 1', 1 / LR, 1000)
print('\n   a (pach/s)   ρ     întârziere')
for a in (500, 800, 900, 950, 990):
    rho = a * LR
    print(f'   {a:8d}   {rho:5.2f}   {LR / (1 - rho) * 1e3:6.0f} ms')
months_double = math.log(0.9 / 0.8) / math.log(1.05)
months_ten = math.log(0.98 / 0.8) / math.log(1.05)
check('3. luni până se dublează întârzierea', months_double, 2.4, rel=0.03)
check('3. luni până întârzierea crește de zece ori', months_ten, 4.2, rel=0.03)

## B.5 Problema 5 — legea lui Little `N = λ × T`
O și **verificăm** pe o coadă simulată: numărul mediu în timp de pachete din sistem, măsurat independent, este egal cu λ × T.

In [ ]:
check('1. timpul în buffer (s)', 40 / 2000, 0.020)
check('2. cereri în curs', 300 * 0.05, 15)
check('3. debitul cu 64 KB în zbor, RTT 20 ms (Mbit/s)', 65536 * 8 / 0.02 / 1e6, 26.2, rel=0.01)

wait, sojourn, arr = mm1_simulate(0.8, n=100_000, service_mean=1e-3)
deps = arr + sojourn
events = sorted([(t, +1) for t in arr] + [(t, -1) for t in deps])
area, n_in, last = 0.0, 0, events[0][0]
for t, step in events:
    area += n_in * (t - last); n_in += step; last = t
N_measured = area / (deps.max() - arr[0])
lam = len(arr) / (arr[-1] - arr[0])
print(f'\nSimulare (ρ = 0,8): N măsurat = {N_measured:.3f}   λ × T = {lam:.0f}/s × {sojourn.mean()*1e3:.3f} ms = {lam * sojourn.mean():.3f}')

## B.6 Problema 6 — nucleu partajat (servere 20 Mbit/s, clienți 8 Mbit/s, 10 transferuri împart o legătură de nucleu R)

In [ ]:
def flow_throughput(R_core, n=10, server=20*M, client=8*M):
    return min(server, client, R_core / n)
check('1. R = 1 Gbit/s (Mbit/s)', flow_throughput(1*G) / M, 8)
check('2. R = 50 Mbit/s (Mbit/s)', flow_throughput(50*M) / M, 5)
print('3. nucleul este gâtuirea când R/10 < 8 Mbit/s, adică R <', 8 * 10, 'Mbit/s')
Rs = np.linspace(10, 200, 400) * M
plt.plot(Rs / M, [flow_throughput(r) / M for r in Rs]); plt.axvline(80, ls=':', color='r')
plt.xlabel('rata legăturii de nucleu R (Mbit/s)'); plt.ylabel('debit per transfer (Mbit/s)'); plt.show()

## B.7 Problema 7 — produsul bandă–întârziere

In [ ]:
for name, R_, rtt, exp in [('(a) LAN de campus', 1*G, 0.0005, 62.5e3), ('(b) București–Frankfurt', 1*G, 0.030, 3.75e6),
                           ('(c) satelit geostaționar', 50*M, 0.600, 3.75e6)]:
    check(f'1. {name} BDP (octeți)', bdp_bytes(R_, rtt), exp)
check('2. fereastră de 64 KB pe calea (b) (Mbit/s)', window_limited(65535, 0.030, 1*G) / M, 17.5, rel=0.01)
print('   = {:.1%} din legătură'.format(window_limited(65535, 0.030, 1*G) / G))
print('3. fereastra necesară pentru a umple (b): cel puțin un BDP = 3,75 MB → opțiunea TCP window scale (RFC 7323)')

## B.8 Problema 8 — argumentul capăt-la-capăt (discuție)
1. Routerele verifică fiecare sumă de control TCP, ca gazdele să nu mai trebuiască. 2. O legătură radio retransmite local cadrele pierdute. 3. Un proxy al ISP-ului recomprimă imaginile. 4. Băncile folosesc TLS capăt-la-capăt peste o rețea deja criptată pe fiecare legătură.

<details><summary>Arată răspunsurile din seminar</summary>

1. **Inconsecvent** ca înlocuitor — corupția poate apărea în interiorul routerelor și gazdelor; doar o verificare capăt-la-capăt o prinde (A.7 de mai sus arată asta).
2. **Consecvent** — o îmbunătățire de performanță; TCP verifică în continuare capăt-la-capăt.
3. **Problematic** — rețeaua modifică date pe care capetele nu i-au cerut să le modifice; anulat de HTTPS.
4. **Consecvent** — doar capetele pot garanta confidențialitatea dintre ele.
</details>

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **P8.1:** 
- **P8.2:** 
- **P8.3:** 
- **P8.4:** 

**Temă**: (1) exercițiile de la curs — comparați cu A.9; (2) decodați de mână primii 54 de octeți ai unui TCP SYN capturat de voi — făcut pentru voi la sfârșitul **Sarcinii 3** de mai jos, verificați fiecare câmp; (3) cinci rânduri despre dacă argumentul Saltzer–Reed–Clark se aplică livrării de e-mail.

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Tema 3 — argumentul capăt-la-capăt și e-mailul:** 

---
# Partea C · Laboratorul 2 (Linux) — analiza pachetelor cu tcpdump, tshark și Scapy
Colab nu are un desktop grafic, așa că **cele trei panouri ale lui Wireshark sunt înlocuite de cod**: `tcpdump` capturează, `tshark` aplică filtrele de afișare și statisticile lui Wireshark, iar Scapy ne permite să deschidem fiecare antet. Fiecare fișier de captură este salvat și îl puteți descărca (ultima celulă a acestei părți) și deschide în Wireshark pe propriul calculator.

In [ ]:
from scapy.all import rdpcap, Ether, IP, TCP, UDP, ICMP, DNS, Raw, IPerror, UDPerror, TCPerror
IFACE, GATEWAY, MY_IP = default_route()
WEB_HOST = 'example.org'
print(f'Captură pe {IFACE}; adresa mea {MY_IP}; gateway {GATEWAY}')

def capture(action, out, bpf='', iface=None, pre=1.5, post=1.5):
    """Pornește tcpdump, rulează action(), oprește tcpdump. Întoarce lista de pachete (Scapy)."""
    cmd = f'{SUDO}tcpdump -i {iface or IFACE} -n -U -s 0 -w {out} {bpf}'
    p = subprocess.Popen(shlex.split(cmd), stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    time.sleep(pre)
    try:
        action()
    except Exception as e:
        print('acțiunea a eșuat:', e)
    time.sleep(post)
    p.send_signal(signal.SIGINT)
    try:
        err = p.communicate(timeout=10)[1]
    except subprocess.TimeoutExpired:
        p.kill(); err = p.communicate()[1]
    summary = [l for l in err.splitlines() if 'captured' in l or 'error' in l.lower()]
    print(f'[{out}]', '; '.join(summary) or err.strip()[-200:])
    try:
        return rdpcap(out)
    except Exception as e:
        print('nu pot citi captura:', e); return []

def question(d):
    """Prima înregistrare de întrebare a unui mesaj DNS (merge cu orice versiune Scapy)."""
    qd = d.qd
    if not qd: return None
    return qd[0] if isinstance(qd, list) else qd     # o listă în Scapy ≥ 2.6, o singură înregistrare înainte

def info(p):
    """O descriere pe un rând, ca în coloana Info din Wireshark."""
    if p.haslayer(DNS):
        d = p[DNS]; qr = question(d)
        q = qr.qname.decode() if qr else '?'
        qt = {1: 'A', 28: 'AAAA', 5: 'CNAME', 65: 'HTTPS'}.get(qr.qtype, qr.qtype) if qr else ''
        return f'DNS {"răspuns" if d.qr else "interogare"} {qt} {q}'
    if p.haslayer(TCP):
        t = p[TCP]; fl = str(t.flags)
        names = {'S': 'SYN', 'SA': 'SYN-ACK', 'A': 'ACK', 'FA': 'FIN-ACK', 'PA': 'PSH-ACK', 'R': 'RST', 'RA': 'RST-ACK', 'F': 'FIN', 'FPA': 'FIN-PSH-ACK'}
        s = f'TCP {t.sport}→{t.dport} [{names.get(fl, fl)}] len={len(t.payload)}'
        if p.haslayer(Raw):
            load = p[Raw].load
            if load[:4] in (b'GET ', b'POST', b'HEAD'):
                s = 'HTTP ' + load.split(b'\r\n')[0].decode(errors='replace')
            elif load[:5] == b'HTTP/':
                s = 'HTTP ' + load.split(b'\r\n')[0].decode(errors='replace')
            elif load[:1] == b'\x16' and len(load) > 5:
                s = 'TLS ' + {1: 'Client Hello', 2: 'Server Hello'}.get(load[5], 'handshake')
            elif load[:1] == b'\x17':
                s = 'TLS date de aplicație (criptate)'
        return s
    if p.haslayer(ICMP):
        i = p[ICMP]
        return f'ICMP {dict([(0,"Echo Reply"),(3,"Destination Unreachable"),(8,"Echo Request"),(11,"Time Exceeded")]).get(i.type, i.type)} (tipul {i.type}, codul {i.code})'
    if p.haslayer(UDP):
        return f'UDP {p[UDP].sport}→{p[UDP].dport}'
    return p.summary()

def protocol(p):
    for layer, name in [(DNS, 'DNS'), (ICMP, 'ICMP')]:
        if p.haslayer(layer): return name
    if p.haslayer(TCP):
        port = min(p[TCP].sport, p[TCP].dport)
        return {80: 'HTTP/TCP', 443: 'TLS/TCP'}.get(port, 'TCP')
    if p.haslayer(UDP): return 'UDP'
    if p.haslayer(IP): return 'IP'
    return p.lastlayer().name

def packet_list(pkts, flt=lambda p: True, limit=40):
    t0 = pkts[0].time if pkts else 0
    shown = 0
    print(f'{"Nr.":>4s} {"Timp":>8s}  {"Sursa":15s} {"Destinația":15s} {"Protocol":9s} Info')
    for i, p in enumerate(pkts, 1):
        if not flt(p): continue
        src = p[IP].src if p.haslayer(IP) else '?'; dst = p[IP].dst if p.haslayer(IP) else '?'
        print(f'{i:4d} {float(p.time - t0):8.4f}  {src:15s} {dst:15s} {protocol(p):9s} {info(p)}')
        shown += 1
        if shown >= limit: print('...'); break

## Sarcina 1 — O primă captură

In [ ]:
first = capture(lambda: sh(f'curl -4 -s http://{WEB_HOST}/ -o /dev/null'), 'first.pcap')
packet_list(first, limit=20)

In [ ]:
print('Total pachete :', len(first))
print('Pe protocoale :', dict(Counter(protocol(p) for p in first)))
server_ips = {p[IP].dst for p in first if p.haslayer(TCP) and p[TCP].dport == 80}
mine = [p for p in first if p.haslayer(DNS) or (p.haslayer(IP) and (p[IP].src in server_ips or p[IP].dst in server_ips))]
print('Aparțin lui curl (DNS + serverul web):', len(mine), '\n')
ids = {id(x) for x in mine}
packet_list(first, lambda p: id(p) in ids)

# Panoul „Packet details”: straturile unui pachet, de la exterior spre interior
if mine:
    p = next((x for x in mine if x.haslayer(Raw)), mine[0])
    layer, names = p, []
    while layer:
        names.append(layer.name); layer = layer.payload
    print('\nStraturile unui pachet:', ' / '.join(names))
    p.show()

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q1 — numărul total de pachete; câte aparțin lui curl; ce altceva mai trimite mașina?:** 
- **Q2 — straturile afișate și corespondența lor cu modelul cu 5 straturi:** 

## Sarcina 2 — Filtre de captură și filtre de afișare
| | Filtru de captură | Filtru de afișare |
|---|---|---|
| Când acționează | înainte de captură: pachetele care nu se potrivesc nu sunt înregistrate niciodată | după captură: pachetele care nu se potrivesc sunt doar ascunse |
| Sintaxa | BPF, ca în tcpdump | limbajul de câmpuri al lui Wireshark |
| Exemple | `host example.org` · `port 53` · `tcp port 80` · `icmp` | `dns` · `http` · `tcp.port == 80` · `icmp` · `tcp.flags.syn == 1` |

In [ ]:
filtered = capture(lambda: sh(f'curl -4 -s http://{WEB_HOST}/ -o /dev/null', show=False),
                   'filtered.pcap', bpf='port 53 or port 80')
print(f'Cu filtrul de captură au fost înregistrate doar {len(filtered)} pachete.\n')
DISPLAY_FILTERS = ['dns', 'http', 'tcp.port == 80', 'tcp.flags.syn == 1', 'dns.flags.response == 1', f'ip.src == {MY_IP}']
for flt in DISPLAY_FILTERS:
    if have('tshark'):
        out = sh(f"tshark -r filtered.pcap -Y '{flt}' 2>/dev/null", show=False)
        print(f'--- filtru de afișare: {flt}   ({len(out.strip().splitlines())} pachete)')
        print('\n'.join(out.strip().splitlines()[:6]))

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q3 — filtru de afișare doar pentru răspunsuri DNS; filtru de afișare pentru pachetele trimise de mașina voastră:** 

## Sarcina 3 — O cerere web, strat cu strat
Cererea trebuie să înceapă cu o interogare DNS reală. Dacă acest VM folosește un resolver local cu cache (`127.0.0.x` în `/etc/resolv.conf`), interogarea lui nu ar apărea pe interfața de rețea, așa că notebook-ul face atunci pasul DNS explicit, către un resolver public, chiar înainte de cererea HTTP.

In [ ]:
ns = re.findall(r'^nameserver\s+(\S+)', open('/etc/resolv.conf').read(), re.M)
local_resolver = not ns or ipaddress.ip_address(ns[0]).is_loopback
print('Resolver(e):', ns, '→', 'cache local, pasul DNS se face explicit' if local_resolver else 'interogările ies pe interfață')
if have('resolvectl'):
    sh(f'{SUDO}resolvectl flush-caches', show=False)

def web_request():
    if local_resolver:
        addr = sh(f'dig +short A {WEB_HOST} @8.8.8.8 | tail -1', show=False).strip()
        sh(f'curl -4 -s --resolve {WEB_HOST}:80:{addr} http://{WEB_HOST}/ -o /dev/null', show=False)
    else:
        sh(f'curl -4 -s http://{WEB_HOST}/ -o /dev/null', show=False)

web = capture(web_request, 'web.pcap')
if have('editcap'):
    sh('editcap -F pcapng web.pcap web.pcapng', show=False); print('am salvat web.pcapng')
web_server = next((p[IP].dst for p in web if p.haslayer(TCP) and p[TCP].dport == 80 and str(p[TCP].flags) == 'S'), None)
dns_server = next((p[IP].dst for p in web if p.haslayer(DNS) and not p[DNS].qr), None)
print('server web:', web_server, '| server DNS:', dns_server, '\n')
conv = lambda p: p.haslayer(DNS) or (p.haslayer(TCP) and web_server in (p[IP].src, p[IP].dst))
packet_list(web, conv)

In [ ]:
# Completăm tabelul din fișa de laborator din pachetul HTTP GET
get = next((p for p in web if p.haslayer(Raw) and p[Raw].load.startswith(b'GET ')), None)
if get is None:
    print('Nu s-a găsit niciun HTTP GET — a fost situl redirecționat la HTTPS? Folosiți exact http://.')
else:
    e, i, t = get[Ether], get[IP], get[TCP]
    lines = get[Raw].load.split(b'\r\n')
    host = next((l.decode() for l in lines if l.lower().startswith(b'host:')), '')
    rows = [('Ethernet', 'MAC sursă / MAC destinație', f'{e.src} / {e.dst}'),
            ('Ethernet', 'Tip', f'0x{e.type:04x} ({ETHERTYPE.get(e.type)})'),
            ('IPv4', 'Adresa sursă / destinație', f'{i.src} / {i.dst}'),
            ('IPv4', 'TTL · Protocol · Lungime totală', f'{i.ttl} · {i.proto} ({IPPROTO.get(i.proto)}) · {i.len}'),
            ('TCP', 'Port sursă / Port destinație', f'{t.sport} / {t.dport}'),
            ('TCP', 'Număr de secvență · Lungime antet', f'{t.seq} · {t.dataofs * 4} octeți'),
            ('HTTP', 'Linia de cerere · Antetul Host', f'{lines[0].decode()} · {host}')]
    for layer, field, value in rows:
        print(f'{layer:9s} {field:34s} {value}')

In [ ]:
# Q5 — MAC-ul destinație al GET-ului vs. MAC-ul gateway-ului
sh(f'ip neigh show {GATEWAY}')
if get is not None:
    gw_mac = re.search(r'lladdr (\S+)', sh(f'ip neigh show {GATEWAY}', show=False))
    print('MAC-ul destinație al GET-ului:', get[Ether].dst, '| MAC-ul gateway-ului:', gw_mac.group(1) if gw_mac else '?')
# Q6 — TTL-ul răspunsului serverului
reply = next((p for p in web if p.haslayer(TCP) and p[IP].src == web_server), None)
if reply is not None:
    ttl = reply[IP].ttl; start = min(x for x in (64, 128, 255) if x >= ttl)
    print(f"TTL-ul serverului {ttl} → a pornit probabil de la {start} → circa {start - ttl} routere pe calea de întoarcere")
# Q7 — octeții de antet dinaintea primului octet HTTP
if get is not None:
    hdr_bytes = len(get) - len(get[Raw].load)
    print(f'{hdr_bytes} octeți de antet = Ethernet 14 + IP {get[IP].ihl*4} + TCP {get[TCP].dataofs*4} '
          f'(antetul TCP poartă {get[TCP].dataofs*4 - 20} octeți de opțiuni, de ex. marcaje de timp)')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q4 — cheile de demultiplexare; ce s-ar întâmpla dacă EtherType ar fi 0x86DD?:** 
- **Q5 — MAC-ul destinație vs. MAC-ul gateway-ului: explicați:** 
- **Q6 — TTL și numărul de routere; de ce pot diferi cele două sensuri?:** 
- **Q7 — octeții de antet dinaintea HTTP, vs. Cursul 2 §3.1:** 

### Temă: decodați de mână propriul SYN
Primii octeți ai SYN-ului pe care tocmai l-ați trimis, decodați cu decodorul din Seminar. Verificați fiecare câmp față de ieșirea `tshark -V` de dedesubt.

In [ ]:
syn = next((p for p in web if p.haslayer(TCP) and str(p[TCP].flags) == 'S'), None)
if syn is not None:
    raw = bytes(syn)
    hexdump(raw[:54])
    e_, rest = decode_ethernet(raw); i_, rest = decode_ipv4(rest); t_, _ = decode_tcp(rest)
    print('\nEthernet:', e_); print('IPv4    :', i_); print('TCP     :', t_)
    print(f'\nSYN-ul are {len(raw)} octeți: 54 de octeți de antete fixe + {t_["options_len"]} octeți de opțiuni TCP (MSS, SACK, marcaje de timp, window scale).')
    if have('tshark'):
        idx = web.index(syn) + 1
        sh(f"tshark -r web.pcap -Y 'frame.number == {idx}' -V 2>/dev/null | sed -n '1,80p'")

## Sarcina 4 — Întreaga conversație
*Follow TCP stream*, *Protocol hierarchy*, *Conversations* și *Flow graph*, fără interfața grafică.

In [ ]:
def follow_tcp_stream(pkts, server, port=80):
    parts = {'client→server': {}, 'server→client': {}}
    for p in pkts:
        if p.haslayer(TCP) and p.haslayer(Raw) and server in (p[IP].src, p[IP].dst) and port in (p[TCP].sport, p[TCP].dport):
            d = 'client→server' if p[IP].dst == server else 'server→client'
            parts[d][p[TCP].seq] = p[Raw].load
    for d, segs in parts.items():
        data = b''.join(segs[k] for k in sorted(segs))
        print(f'===== {d} ({len(data)} octeți) =====')
        print(data[:700].decode('latin-1')); print()
follow_tcp_stream(web, web_server)

In [ ]:
if have('tshark'):
    sh("tshark -r web.pcap -q -z io,phs 2>/dev/null")      # Statistics → Protocol Hierarchy (ierarhia protocoalelor)
    sh("tshark -r web.pcap -q -z conv,ip 2>/dev/null")     # Statistics → Conversations (conversații)

In [ ]:
def flow_graph(pkts, client, peers, title):
    """Statistics → Flow Graph: o diagramă timp–secvență."""
    names = {client: f'client\n{client}'}
    cols = [client] + [p for p in peers if p]
    for pr in peers:
        if pr: names[pr] = ('server DNS' if pr == dns_server else 'server web') + f'\n{pr}'
    x = {h: i for i, h in enumerate(cols)}
    sel = [p for p in pkts if p.haslayer(IP) and p[IP].src in x and p[IP].dst in x and p[IP].src != p[IP].dst]
    if not sel:
        print('nimic de desenat'); return
    t0 = sel[0].time
    fig, ax = plt.subplots(figsize=(10, 0.45 * len(sel) + 1.5))
    for h in cols:
        ax.axvline(x[h], color='gray', lw=1)
        ax.text(x[h], -0.8, names[h], ha='center', va='bottom', fontsize=8, fontweight='bold')
    for k, p in enumerate(sel):
        a, z = x[p[IP].src], x[p[IP].dst]
        ax.annotate('', xy=(z, k), xytext=(a, k), arrowprops=dict(arrowstyle='->', color='tab:blue' if a < z else 'tab:green'))
        ax.text((a + z) / 2, k - 0.15, info(p), ha='center', fontsize=7)
        ax.text(-0.45, k, f'{float(p.time - t0)*1e3:7.1f} ms', fontsize=7, va='center')
    ax.set_ylim(len(sel), -1.2); ax.set_xlim(-0.6, len(cols) - 0.6); ax.axis('off'); ax.set_title(title)
    plt.tight_layout(); plt.show()
    print(f'{len(sel)} pachete pentru o pagină mică')

cols_order = [dns_server, web_server] if dns_server else [web_server]
flow_graph(web, MY_IP, cols_order, f'O cerere web către {WEB_HOST}')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q8 — etichetați graficul de flux: interogare DNS, handshake, cerere, răspuns, închidere; câte pachete pentru o pagină?:** 

## Sarcina 5 — Măsurarea RTT din handshake
Timpul de la SYN-ul vostru la SYN-ACK-ul serverului este un RTT. Wireshark îl calculează și pentru fiecare confirmare (`tcp.analysis.ack_rtt`).

In [ ]:
syn = next((p for p in web if p.haslayer(TCP) and str(p[TCP].flags) == 'S'), None)
synack = next((p for p in web if p.haslayer(TCP) and str(p[TCP].flags) == 'SA'), None)
if syn is not None and synack is not None:
    print(f'SYN → SYN-ACK: {float(synack.time - syn.time)*1e3:.2f} ms  (RTT-ul handshake-ului)')
if have('tshark'):
    sh("tshark -r web.pcap -Y 'tcp.analysis.ack_rtt' -T fields -e frame.number -e ip.src -e tcp.analysis.ack_rtt 2>/dev/null | head")
out = sh(f'ping -c 5 -W 2 {WEB_HOST}', timeout=20)
if ' 0 received' in out or 'received' not in out:
    print('ICMP blocat sau fără răspuns → în schimb, RTT prin conectare TCP:'); tcp_rtt(WEB_HOST, 80, 5)

# DNS: timpul dintre interogare și răspunsul ei
qs = {p[DNS].id: p.time for p in web if p.haslayer(DNS) and not p[DNS].qr}
for p in web:
    if p.haslayer(DNS) and p[DNS].qr and p[DNS].id in qs:
        print(f'DNS {info(p)}: răspuns în {float(p.time - qs[p[DNS].id])*1e3:.2f} ms')

In [ ]:
# Q10 — repetăm FĂRĂ a goli vreun cache: mai există o interogare DNS?
again = capture(lambda: sh(f'curl -4 -s http://{WEB_HOST}/ -o /dev/null', show=False), 'again.pcap', bpf='port 53 or port 80')
n_dns = sum(1 for p in again if p.haslayer(DNS))
print(f'Pachete DNS a doua oară: {n_dns}. ' + ('Fără un resolver cu cache pe această gazdă, fiecare program întreabă din nou.' if n_dns
      else 'Nicio interogare: răspunsul a venit dintr-un cache local, valabil pe durata TTL-ului înregistrării DNS.'))

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q9 — RTT-ul din handshake vs. ping; explicați orice diferență:** 
- **Q10 — timpul de răspuns DNS; există o interogare a doua oară și de ce?:** 

## Sarcina 6 — Traceroute la microscop
`traceroute` din Linux trimite probe UDP spre porturi înalte (de la 33434 în sus) cu TTL = 1, 2, 3, … Routerele răspund cu ICMP Time Exceeded și citează începutul probei originale în mesajul lor.

In [ ]:
target_ip = resolve(WEB_HOST)
tr = capture(lambda: sh(f'traceroute -n -q 1 -w 2 -m 6 {WEB_HOST}', timeout=60), 'traceroute.pcap',
             bpf=f'icmp or (udp and dst host {target_ip})' if target_ip else 'icmp or udp')
probes = [p for p in tr if p.haslayer(UDP) and not p.haslayer(ICMP) and p[IP].src == MY_IP and p[UDP].dport >= 33434]
print('Probe trimise:')
for p in probes:
    print(f'   UDP {p[UDP].sport} → {p[IP].dst}:{p[UDP].dport}   TTL = {p[IP].ttl}')
print('Răspunsuri:')
for p in tr:
    if p.haslayer(ICMP) and p[IP].dst == MY_IP:
        quoted = ''
        if p.haslayer(IPerror):
            q = p[IPerror]
            quoted = f'  citează: {q.src} → {q.dst} TTL={q.ttl}' + (f' UDP dport {p[UDPerror].dport}' if p.haslayer(UDPerror) else '')
        print(f'   de la {p[IP].src:15s} {info(p)}{quoted}')
if not any(p.haslayer(ICMP) for p in tr):
    print('   (niciun răspuns ICMP: această rețea cloud le filtrează — încercați din nou de pe o mașină din laborator/de acasă)')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q11 — protocolul, porturile destinație și TTL-ul probelor:** 
- **Q12 — tipul și codul ICMP al răspunsurilor; de ce poartă înăuntru proba originală?:** 

## Sarcina 7 — Captură în linie de comandă: tcpdump și tshark

In [ ]:
def tcpdump_live(args, action, seconds=8):
    p = subprocess.Popen(f'{SUDO}timeout {seconds} tcpdump {args}', shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    time.sleep(1.5)
    try: action()
    except Exception as e: print('acțiunea a eșuat:', e)
    print(p.communicate()[0])

req = lambda: sh(f'curl -4 -s http://{WEB_HOST}/ -o /dev/null', show=False)
print('##### cu -n (doar adrese)'); tcpdump_live(f"-i {IFACE} -n -c 12 'port 53 or port 80'", req)
print('##### fără -n (tcpdump caută numele, ceea ce e mai lent)'); tcpdump_live(f"-i {IFACE} -c 12 'port 80'", req, seconds=15)

In [ ]:
cli = capture(req, 'cli.pcap', bpf=f'host {WEB_HOST}')
if have('tshark'):
    sh('tshark -r cli.pcap 2>/dev/null')
    sh("tshark -r cli.pcap -Y http.request -T fields -e ip.src -e ip.dst -e http.host -e http.request.uri 2>/dev/null")

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q13 — este aceeași conversație ca în Sarcina 3? Ce schimbă -n?:** 

## Sarcina 8 — HTTPS: ce rămâne vizibil
Cererea HTTP este criptată, dar **Client Hello** poartă încă numele serverului în extensia `server_name` (SNI). Mai jos, Client Hello este analizat octet cu octet.

In [ ]:
def client_hello_sni(data):
    """Întoarce server_name dintr-un TLS Client Hello (octeți începând cu înregistrarea TLS)."""
    try:
        if data[0] != 0x16 or data[5] != 0x01: return None
        p = 5 + 4 + 2 + 32                         # antet de înregistrare, antet de handshake, versiune, random
        p += 1 + data[p]                           # id-ul sesiunii
        p += 2 + int.from_bytes(data[p:p+2], 'big')  # suitele de cifrare
        p += 1 + data[p]                           # metodele de compresie
        end = p + 2 + int.from_bytes(data[p:p+2], 'big'); p += 2
        while p + 4 <= min(end, len(data)):
            etype, elen = int.from_bytes(data[p:p+2], 'big'), int.from_bytes(data[p+2:p+4], 'big'); p += 4
            if etype == 0:                         # server_name
                n = int.from_bytes(data[p+3:p+5], 'big'); return data[p+5:p+5+n].decode()
            p += elen
    except IndexError:
        return None

tls = capture(lambda: sh(f'curl -4 -s https://{WEB_HOST}/ -o /dev/null', show=False), 'https.pcap', bpf='port 53 or tcp port 443')

# Găsim conexiunea noastră: un flux care ÎNCEPE (SYN) în timpul capturii și al cărui Client Hello numește WEB_HOST
flows = {}
for p in tls:
    if p.haslayer(TCP) and p.haslayer(IP) and p[IP].src == MY_IP and p[TCP].dport == 443:
        key = (p[IP].dst, p[TCP].sport)
        if str(p[TCP].flags) == 'S': flows.setdefault(key, {})
        if key in flows and p.haslayer(Raw): flows[key][p[TCP].seq] = p[Raw].load
srv443, hello, my_port = None, b'', None
for (dst, sport), segs in flows.items():
    data = b''.join(segs[k] for k in sorted(segs))[:4000]
    if client_hello_sni(data) == WEB_HOST or srv443 is None:
        srv443, hello, my_port = dst, data, sport
conn = lambda p: p.haslayer(DNS) or (p.haslayer(TCP) and srv443 in (p[IP].src, p[IP].dst))
packet_list(tls, conn, limit=25)
print('\nSNI găsit de analizorul nostru:', client_hello_sni(hello))
if have('tshark'):
    sh("tshark -r https.pcap -Y 'tls.handshake.type == 1' -T fields -e ip.dst -e tls.handshake.extensions_server_name 2>/dev/null")

In [ ]:
# Q15 — ce mai află un observator din rețea
names = sorted({question(p[DNS]).qname.decode() for p in tls if p.haslayer(DNS) and question(p[DNS])})
flows = [p for p in tls if p.haslayer(TCP) and srv443 in (p[IP].src, p[IP].dst) and my_port in (p[TCP].sport, p[TCP].dport)]
up = sum(len(p) for p in flows if p[IP].dst == srv443); down = sum(len(p) for p in flows if p[IP].src == srv443)
print('Observatorul vede:')
print(' • numele căutat (DNS, în clar):', names)
print(' • numele serverului în Client Hello (SNI):', client_hello_sni(hello))
print(' • adrese și porturi:', MY_IP, '↔', f'{srv443}:443')
print(f' • dimensiuni: {up} octeți trimiși, {down} octeți primiți, {len(flows)} pachete')
if flows: print(f' • momente: vizita a durat {float(flows[-1].time - flows[0].time)*1e3:.0f} ms')
print('Observatorul NU vede: calea din URL, antetele HTTP, cookie-urile, conținutul paginii.')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q14 — puteți citi cererea? Ce conține server_name?:** 
- **Q15 — ce mai poate afla un observator:** 
- **Q16 — ce straturi criptează TLS și care circulă în clar; legătura cu argumentul capăt-la-capăt:** 

### Descărcați fișierele de captură (pentru a le deschide în Wireshark pe propriul calculator)

In [ ]:
files_to_get = [f for f in ['web.pcapng', 'web.pcap', 'traceroute.pcap', 'https.pcap', 'cli.pcap'] if os.path.exists(f)]
print('Disponibile:', files_to_get)
if IN_COLAB:
    from google.colab import files
    for f in files_to_get:
        if f != 'web.pcap' or 'web.pcapng' not in files_to_get:
            files.download(f)

---
# Partea D · Laboratorul 2 (Windows) — Wireshark și pktmon, văzute de aici
Wireshark este același program pe Windows; diferă driverul de captură (**Npcap**), unealta de captură în linie de comandă integrată (**pktmon**, doar pentru administrator) și **`tracert`**, care trimite ICMP Echo Request în loc de UDP. Ultima diferență o putem reproduce exact: `traceroute -I` trimite probe ca `tracert`.

## Sarcina 6 (Windows) — `tracert` la microscop
`tracert -d -h 6 -w 1000 example.org` trimite **3 ICMP Echo Request pe hop**; destinația răspunde cu Echo Reply.

In [ ]:
tw = capture(lambda: sh(f'{SUDO}traceroute -I -n -q 3 -w 1 -m 6 {WEB_HOST}', timeout=60), 'tracert_style.pcap', bpf='icmp')
echo = [p for p in tw if p.haslayer(ICMP) and p[ICMP].type == 8 and p[IP].src == MY_IP]
print(f'{len(echo)} ICMP Echo Request trimise; TTL-uri:', Counter(p[IP].ttl for p in echo))
print('Tipuri de răspuns:', Counter(info(p) for p in tw if p.haslayer(ICMP) and p[IP].dst == MY_IP))
print('\nComparați cu traceroute din Linux în Sarcina 6: UDP spre porturi ≥ 33434 → destinația răspunde Port Unreachable.')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q11 (Windows) — tipul pachetelor, TTL pe hop, probe pe hop; diferența față de traceroute din Linux:** 
- **Q12 (Windows) — tipul ICMP al răspunsurilor și ce este citat înăuntru:** 

## Linux și Windows în paralel

In [ ]:
SIDE = [('Driver de captură', 'nucleul (libpcap)', 'Npcap'),
        ('Captură grafică', 'Wireshark', 'Wireshark'),
        ('Captură integrată în linie cmd.', 'tcpdump', 'pktmon (admin)'),
        ('Analiză în linie de comandă', 'tshark', 'tshark.exe'),
        ('Golirea cache-ului DNS', 'resolvectl flush-caches', 'ipconfig /flushdns'),
        ('Probe traceroute', 'UDP spre porturi înalte (implicit)', 'ICMP Echo Request'),
        ('TTL inițial tipic', '64', '128')]
print(f'{"Scop":31s} {"Linux":35s} Windows')
for row in SIDE: print(f'{row[0]:31s} {row[1]:35s} {row[2]}')
print('\nTTL-ul inițial al acestui VM Linux:', open('/proc/sys/net/ipv4/ip_default_ttl').read().strip())

### pktmon, pentru mașina voastră Windows
```powershell
# (admin)
pktmon start --capture --comp nics --pkt-size 0 -f cli.etl
curl.exe -s http://example.org/ -o NUL
pktmon stop
pktmon etl2pcap cli.etl -o cli.pcapng
& "C:\Program Files\Wireshark\tshark.exe" -r cli.pcapng -Y http.request -T fields -e ip.src -e ip.dst -e http.host -e http.request.uri
```
**Q13 (Windows)** — `pktmon` vede fiecare pachet în mai multe puncte din stiva de rețea, așa că unele pachete apar de mai multe ori. Comparați ieșirea lui cu `cli.pcap` din Sarcina 7, pe care o puteți descărca mai sus.

**Notă Wi-Fi.** Pe un adaptor Wi-Fi, atât Windows cât și Linux (în modul normal) îi dau lui Wireshark cadre care arată ca Ethernet; antetele 802.11 reale apar doar în modul monitor.

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q13 (Windows) — duplicate în pktmon; aceeași conversație ca în Sarcina 3?:** 

---
### Săptămâna viitoare
Cursul 3 deschide stratul aplicație cu Web-ul și HTTP. În Laboratorul 3 scrieți primele programe de rețea: un client și un server TCP folosind socket-uri.